# UNIFESP v2 -- leakage-free, ensemble-ready pipeline (paper protocol + Kaggle test predictions)

**Attach ONE dataset:** `felipekitamura/unifesp-xray-bodypart-classification`. **Accelerator:** GPU T4 x2. **Internet:** on.

1. Builds `index.csv`, the patient-level 5-fold split, a fixed **20 % held-out set** and a 5-fold **CV pool**.
2. Trains every run on every CV fold (`src/train_v2.py`): fixed schedule + EMA, **no checkpoint selection**, so out-of-fold (OOF)
   predictions are unbiased. Each fold also predicts the held-out set and the Kaggle test set.
3. `src/run_ensemble.py` makes ALL decisions on OOF only, reports the held-out score ONCE with patient-level bootstrap CIs and writes
   `submission.csv` for Kaggle *Late Submission*.

Resumable: finished runs (`preds.npz` exists) are skipped, so after a cut-off session just run all again.

In [1]:
import os
images_root = None
for cur, dirs, files in os.walk("/kaggle/input"):
    if "train" in dirs and "test" in dirs:
        images_root = cur
        dirs[:] = [d for d in dirs if d not in ("train", "test")]
        break
assert images_root is not None, "Attach felipekitamura/unifesp-xray-bodypart-classification (Notebook Settings -> Data)."
print("Images dataset:", images_root)
IMAGES_DATASET = images_root

Images dataset: /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification


## Source files (generated from `experiments/v2/src/` -- do not edit here)

In [2]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/__init__.py", "w", encoding="utf-8") as f:
    f.write('')
print("wrote src/__init__.py")

wrote src/__init__.py


In [3]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/config.py", "w", encoding="utf-8") as f:
    f.write('"""\nCentral config. Two roots, because on Kaggle they\'ll usually be two\ndifferent attached datasets:\n  - DATA_ROOT: where train/, test/ (the DICOM images) live\n  - INDEX_ROOT: where index.csv and splits/folds.csv live\n\nLocally both default to the repo root. On Kaggle, set them explicitly as\nenvironment variables in the notebook\'s first cell, e.g.:\n    import os\n    os.environ["UNIFESP_DATA_ROOT"] = "/kaggle/input/unifesp-xray-bodypart-classification"\n    os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/input/unifesp-index-files"  # your uploaded index.csv+splits/\n    os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working"\n(see KAGGLE_SETUP.md for the full first-cell snippet.)\n"""\nimport os\n\n_THIS_DIR = os.path.dirname(os.path.abspath(__file__))\n_REPO_ROOT = os.path.dirname(_THIS_DIR)\n\nON_KAGGLE = os.path.exists("/kaggle/input")\n\nif ON_KAGGLE:\n    DATA_ROOT = os.environ.get("UNIFESP_DATA_ROOT", "/kaggle/input")\n    INDEX_ROOT = os.environ.get("UNIFESP_INDEX_ROOT", DATA_ROOT)\n    OUTPUT_DIR = os.environ.get("UNIFESP_OUTPUT_DIR", "/kaggle/working")\nelse:\n    DATA_ROOT = os.environ.get("UNIFESP_DATA_ROOT", _REPO_ROOT)\n    INDEX_ROOT = os.environ.get("UNIFESP_INDEX_ROOT", DATA_ROOT)\n    OUTPUT_DIR = os.environ.get("UNIFESP_OUTPUT_DIR", os.path.join(_REPO_ROOT, "runs"))\n\nINDEX_CSV = os.path.join(INDEX_ROOT, "index.csv")\nFOLDS_CSV = os.path.join(INDEX_ROOT, "splits", "folds.csv")\n\n# Optional: where cache_images.py pre-decodes DICOM -> PNG once, so every\n# later training run reads a small PNG instead of re-decoding DICOM every\n# epoch (top Kaggle finishers reported this cuts training from hours to\n# minutes). None (default) means "no cache, decode DICOM every time" --\n# fully backward compatible, nothing breaks if you skip this step.\nCACHE_DIR = os.environ.get("UNIFESP_CACHE_DIR", os.path.join(OUTPUT_DIR, "image_cache") if ON_KAGGLE else None)\n\n# Which dataset this run targets -- "unifesp" (default), "chestxray14",\n# "mura", or "covidcxr". Switching this is the ONLY config change needed to\n# retarget the whole pipeline (train.py/train_hybrid.py/evaluate_on_holdout.py/\n# etc. are all dataset-agnostic): it picks the label set below (-> last-layer\n# neuron count, via build_model\'s/HybridCNNTransformer\'s default n_classes=\n# C.N_CLASSES) and, in dataset.py, the image file format to load (.dcm vs\n# .png). Still set UNIFESP_DATA_ROOT to that dataset\'s own attached path.\nDATASET = os.environ.get("UNIFESP_DATASET", "unifesp")\n\nif DATASET == "chestxray14":\n    # NIH ChestX-ray14: 14 disease classes. "No Finding" is deliberately not\n    # its own class -- it\'s the all-zero label vector (same multilabel\n    # convention the CheXNet-style literature on this dataset uses), which\n    # engine.py\'s sample_mean_f1/compute_metrics already handle correctly.\n    LABEL_NAMES = {\n        0: "Atelectasis", 1: "Cardiomegaly", 2: "Consolidation", 3: "Edema",\n        4: "Effusion", 5: "Emphysema", 6: "Fibrosis", 7: "Hernia",\n        8: "Infiltration", 9: "Mass", 10: "Nodule", 11: "Pleural_Thickening",\n        12: "Pneumonia", 13: "Pneumothorax",\n    }\nelif DATASET == "mura":\n    # Stanford MURA, re-tasked as BODY-PART classification (the body part is\n    # already encoded in each image\'s folder name, XR_ELBOW/XR_HAND/...) --\n    # the same task UNIFESP solves, on an independent patient population and\n    # imaging source. Single-label (one body part per image) rather than\n    # multilabel, but a one-hot row is just a degenerate multilabel row, so\n    # nothing downstream needs to know the difference.\n    LABEL_NAMES = {\n        0: "Elbow", 1: "Finger", 2: "Forearm", 3: "Hand",\n        4: "Humerus", 5: "Shoulder", 6: "Wrist",\n    }\nelif DATASET == "covidcxr":\n    # COVID-19 Radiography Database (Qatar University + University of Dhaka),\n    # re-tasked as 4-class chest X-ray diagnosis classification. Single-label\n    # (one diagnosis per image), aggregated from many source datasets/\n    # publications with no consistent patient-id field across them, so\n    # build_index_covidcxr.py uses each image\'s own uid as its "patient_id"\n    # -- an image-level split, not patient-level (a limitation worth naming\n    # in the paper, same as why the other datasets\' patient-level splitting\n    # matters in the first place).\n    LABEL_NAMES = {\n        0: "COVID", 1: "Lung_Opacity", 2: "Normal", 3: "Viral_Pneumonia",\n    }\nelse:\n    LABEL_NAMES = {\n        0: "Abdomen", 1: "Ankle", 2: "Cervical Spine", 3: "Chest", 4: "Clavicles",\n        5: "Elbow", 6: "Feet", 7: "Finger", 8: "Forearm", 9: "Hand", 10: "Hip",\n        11: "Knee", 12: "Lower Leg", 13: "Lumbar Spine", 14: "Others", 15: "Pelvis",\n        16: "Shoulder", 17: "Sinus", 18: "Skull", 19: "Thigh", 20: "Thoracic Spine",\n        21: "Wrist",\n    }\nN_CLASSES = len(LABEL_NAMES)\nIMG_SIZE = 256\nSEED = 42\n\nos.makedirs(OUTPUT_DIR, exist_ok=True)\n')
print("wrote src/config.py")

wrote src/config.py


In [4]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/dataset.py", "w", encoding="utf-8") as f:
    f.write('"""\nUNIFESP DICOM dataset for multilabel body-part classification.\n\nHandles the two DICOM quirks we found in EDA:\n  - MONOCHROME1 vs MONOCHROME2 photometric interpretation (2367 vs 114 files)\n    -> MONOCHROME1 is inverted, so we flip it to a consistent polarity.\n  - Variable BitsStored (8/10/12/15-bit) -> min-max normalize per image instead\n    of assuming a fixed bit depth.\n"""\nimport os\nimport numpy as np\nimport pydicom\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom . import config as C\n\n\ndef load_cached_or_decode(uid, src_path, cache_dir):\n    """Read uid\'s pre-cached PNG if cache_dir has one; otherwise decode the\n    source image (DICOM for unifesp, PNG passthrough for chestxray14 -- see\n    load_image_for_dataset) and, if cache_dir is set, write the cache PNG\n    for next time. Write is atomic (temp file + os.replace) so two training\n    processes caching the same image concurrently (e.g. two backbones on two\n    GPUs both hitting fold 0\'s validation set) never leaves a half-written\n    file for the other to read -- worst case both just redundantly decode it\n    once.\n    """\n    if cache_dir is not None:\n        png_path = os.path.join(cache_dir, f"{uid}.png")\n        if os.path.exists(png_path):\n            return np.array(Image.open(png_path).convert("RGB"))\n\n    image = load_image_for_dataset(src_path)\n\n    if cache_dir is not None:\n        os.makedirs(cache_dir, exist_ok=True)\n        tmp_path = os.path.join(cache_dir, f".{uid}.{os.getpid()}.tmp.png")\n        Image.fromarray(image).save(tmp_path)\n        os.replace(tmp_path, os.path.join(cache_dir, f"{uid}.png"))\n\n    return image\n\n\ndef load_image_for_dataset(path):\n    """Dispatch on C.DATASET: DICOM for unifesp, plain PNG for chestxray14/mura/covidcxr."""\n    if C.DATASET in ("chestxray14", "mura", "covidcxr"):\n        return load_png_as_image(path)\n    return load_dicom_as_image(path)\n\n\ndef load_png_as_image(path):\n    arr = np.array(Image.open(path).convert("L"))  # ChestX-ray14 PNGs are grayscale (some stored as RGBA)\n    arr = np.stack([arr, arr, arr], axis=-1)  # 1-channel X-ray -> pseudo-RGB for ImageNet backbones\n    return arr.astype(np.uint8)\n\n\ndef load_dicom_as_image(path):\n    ds = pydicom.dcmread(path, force=True)\n    arr = ds.pixel_array.astype(np.float32)\n\n    slope = float(getattr(ds, "RescaleSlope", 1.0))\n    intercept = float(getattr(ds, "RescaleIntercept", 0.0))\n    arr = arr * slope + intercept\n\n    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        arr = arr.max() - arr\n\n    lo, hi = arr.min(), arr.max()\n    arr = (arr - lo) / (hi - lo + 1e-6)\n    arr = np.stack([arr, arr, arr], axis=-1)  # 1-channel X-ray -> pseudo-RGB for ImageNet backbones\n    return (arr * 255.0).astype(np.uint8)\n\n\ndef _basename_uid(path):\n    b = os.path.basename(path)[:-4]  # strip .dcm\n    if b.endswith("-c"):\n        b = b[:-2]\n    return b\n\n\ndef build_uid_to_path(data_root):\n    """Walk data_root once and map uid -> absolute file path. We do this at\n    runtime instead of trusting a stored relative path, because the raw\n    image folders are nested inconsistently (a local zip-extract quirk gave\n    us train/train/train/<id>/..., while the Kaggle-mounted dataset or a\n    re-extraction elsewhere may only have train/<id>/...). Matching by the\n    filename\'s UID instead of a fixed path makes this robust to wherever the\n    raw files actually end up on disk."""\n    if C.DATASET == "chestxray14":\n        lookup = {}\n        for root, _dirs, files in os.walk(data_root):\n            for f in files:\n                # skip macOS AppleDouble metadata files (e.g. "._image1.png") --\n                # they match the .png extension but aren\'t real images, and\n                # crash PIL with UnidentifiedImageError (seen on the Kaggle\n                # mirror of this dataset, which was zipped from a Mac)\n                if f.lower().endswith(".png") and not f.startswith("._"):\n                    lookup[f[:-4]] = os.path.join(root, f)\n        return lookup\n\n    if C.DATASET in ("mura", "covidcxr"):\n        # MURA\'s own filenames (image1.png, image2.png, ...) repeat in every\n        # study folder, so the bare filename can\'t be a uid -- use the full\n        # path relative to data_root instead (unique, and keeps the\n        # body-part/class-folder info parseable by build_index_mura.py /\n        # build_index_covidcxr.py). covidcxr also has a parallel "masks/"\n        # tree of lung-segmentation masks with the same filenames as\n        # "images/" -- skip it here so masks never get treated as data.\n        lookup = {}\n        for root, _dirs, files in os.walk(data_root):\n            if C.DATASET == "covidcxr" and os.path.basename(root).lower() == "masks":\n                continue\n            for f in files:\n                # skip macOS AppleDouble metadata files (e.g. "._image1.png") --\n                # they match the .png extension but aren\'t real images, and\n                # crash PIL with UnidentifiedImageError (seen on the Kaggle\n                # mirror of MURA, which was zipped from a Mac)\n                if f.lower().endswith(".png") and not f.startswith("._"):\n                    full_path = os.path.join(root, f)\n                    rel = os.path.relpath(full_path, data_root)\n                    uid = rel.replace(os.sep, "__")[:-4]\n                    lookup[uid] = full_path\n        return lookup\n\n    lookup = {}\n    for root, _dirs, files in os.walk(data_root):\n        for f in files:\n            if f.lower().endswith(".dcm") and not f.startswith("._"):\n                lookup[_basename_uid(f)] = os.path.join(root, f)\n    return lookup\n\n\nclass UnifespDataset(Dataset):\n    def __init__(self, df, data_root, transform=None, has_labels=True, uid_to_path=None, cache_dir=C.CACHE_DIR):\n        """\n        df: a slice of index.csv (optionally merged with folds.csv), must have a \'uid\' column\n        data_root: base dir containing the (possibly nested) train/test DICOM folders\n        uid_to_path: pass a precomputed dict to avoid re-walking data_root for every split\n        cache_dir: optional PNG cache (see load_cached_or_decode); None disables caching\n        """\n        self.df = df.reset_index(drop=True)\n        self.data_root = data_root\n        self.transform = transform\n        self.has_labels = has_labels\n        self.label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(\' \', \'_\')}" for k in range(C.N_CLASSES)]\n        self.uid_to_path = uid_to_path if uid_to_path is not None else build_uid_to_path(data_root)\n        self.cache_dir = cache_dir\n\n        missing = set(self.df["uid"]) - set(self.uid_to_path)\n        if missing:\n            raise FileNotFoundError(\n                f"{len(missing)} uid(s) in this split have no matching .dcm file under {data_root} "\n                f"(e.g. {next(iter(missing))}). Check UNIFESP_DATA_ROOT points at the right dataset."\n            )\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n        uid = row["uid"]\n        path = self.uid_to_path[uid]\n        image = load_cached_or_decode(uid, path, self.cache_dir)\n\n        if self.transform is not None:\n            image = self.transform(image=image)["image"]\n\n        if self.has_labels:\n            target = row[self.label_cols].to_numpy(dtype=np.float32)\n            target = torch.from_numpy(target)\n            return image, target\n        return image, row["uid"]\n\n\ndef get_transforms(img_size=C.IMG_SIZE, train=True):\n    """Train-time augmentation.\n\n    We tried matching the UNIFESP Kaggle competition\'s top finishers\n    (2nd place writeup used GaussNoise + GaussianBlur + ShiftScaleRotate +\n    brightness/contrast) and measured it head-to-head on our own fold 0:\n\n        backbone          light-aug  heavy-aug(+noise/blur)\n        efficientnetv2_s     0.876      0.851   (worse)\n        efficientnet_b3      0.866      0.847   (worse)\n        inception_v3         0.872      0.841   (worse)\n        convnext_tiny        0.738      0.558   (much worse)\n        densenet121          0.869      0.873   (~same)\n        vgg16                0.827      0.836   (~same)\n\n    GaussNoise/GaussianBlur made 6/8 backbones worse (macro_f1), ConvNeXt-\n    Tiny catastrophically so -- likely because blurring removes the fine\n    bone-edge texture that separates visually similar classes (Lumbar vs\n    Thoracic Spine, Hand vs Wrist), and our fixed 30-epoch budget (no\n    warmup, no early stopping) doesn\'t give heavier augmentation enough\n    iterations to pay off the way it did for the competition\'s longer,\n    early-stopped fastai runs. So: geometric augmentation (flip, mild\n    shift/scale/rotate) + brightness/contrast, no synthetic noise/blur.\n    This negative result is itself worth keeping for the paper\'s ablation\n    section.\n    """\n    import albumentations as A\n    from albumentations.pytorch import ToTensorV2\n\n    mean = (0.485, 0.456, 0.406)\n    std = (0.229, 0.224, 0.225)\n\n    if train:\n        return A.Compose([\n            A.Resize(img_size, img_size),\n            A.HorizontalFlip(p=0.5),\n            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),\n            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),\n            A.Normalize(mean=mean, std=std),\n            ToTensorV2(),\n        ])\n    return A.Compose([\n        A.Resize(img_size, img_size),\n        A.Normalize(mean=mean, std=std),\n        ToTensorV2(),\n    ])\n')
print("wrote src/dataset.py")

wrote src/dataset.py


In [5]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/build_index.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1a: Build a flat index of the UNIFESP dataset.\n\nReads every DICOM\'s header (no pixel decode, so it\'s fast) and writes a\nsingle flat CSV that the rest of the pipeline uses. Uses config.DATA_ROOT /\nconfig.INDEX_ROOT (not a path relative to this file) so it works no matter\nwhere the raw images or the output location actually are -- e.g. run fresh\ninside a Kaggle notebook against the attached images dataset, writing to\n/kaggle/working, with no separately-uploaded "index" dataset needed at all.\n\nOutput: <INDEX_ROOT>/index.csv with columns:\n    uid, split, patient_id, study_uid, series_uid, labels, label_0_Abdomen, ...\n(no filepath column -- dataset.py locates files by uid via build_uid_to_path,\nwhich is robust to whatever nesting the raw folders happen to have.)\n"""\nimport os\nimport pydicom\nimport pandas as pd\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path\n\nLABEL_NAMES = C.LABEL_NAMES\nN_CLASSES = C.N_CLASSES\n\n\ndef main():\n    print("Indexing DICOM files under", C.DATA_ROOT, "...")\n    train_uid_to_path = build_uid_to_path(os.path.join(C.DATA_ROOT, "train"))\n    test_uid_to_path = build_uid_to_path(os.path.join(C.DATA_ROOT, "test"))\n    print(f"train dcm files: {len(train_uid_to_path)} | test dcm files: {len(test_uid_to_path)}")\n\n    rows = []\n    for split, uid_to_path in (("train", train_uid_to_path), ("test", test_uid_to_path)):\n        for uid, path in uid_to_path.items():\n            try:\n                ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)\n                patient_id = str(getattr(ds, "PatientID", ""))\n                study_uid = str(getattr(ds, "StudyInstanceUID", ""))\n                series_uid = str(getattr(ds, "SeriesInstanceUID", ""))\n            except Exception as e:\n                patient_id = study_uid = series_uid = ""\n                print("WARN: failed to read header for", path, e)\n            rows.append({\n                "uid": uid, "split": split,\n                "patient_id": patient_id, "study_uid": study_uid, "series_uid": series_uid,\n            })\n\n    index_df = pd.DataFrame(rows)\n\n    # attach multilabel targets for the train split\n    train_csv = pd.read_csv(os.path.join(C.DATA_ROOT, "train.csv"))\n    train_csv["SOPInstanceUID"] = train_csv["SOPInstanceUID"].str.strip()\n    label_map = dict(zip(train_csv["SOPInstanceUID"], train_csv["Target"]))\n    index_df["labels"] = index_df["uid"].map(label_map)\n\n    # one-hot columns for the 22 classes (NaN/absent for test rows)\n    for k in range(N_CLASSES):\n        col = f"label_{k}_{LABEL_NAMES[k].replace(\' \', \'_\')}"\n        index_df[col] = index_df["labels"].apply(\n            lambda s, k=k: int(k in [int(x) for x in str(s).split()]) if pd.notna(s) else pd.NA\n        )\n\n    os.makedirs(C.INDEX_ROOT, exist_ok=True)\n    out_path = os.path.join(C.INDEX_ROOT, "index.csv")\n    index_df.to_csv(out_path, index=False)\n    print("Saved index to", out_path, "| rows:", len(index_df))\n\n    # ---- patient-level leakage sanity check ----\n    train_idx = index_df[index_df["split"] == "train"]\n    test_idx = index_df[index_df["split"] == "test"]\n    train_patients = set(train_idx["patient_id"]) - {""}\n    test_patients = set(test_idx["patient_id"]) - {""}\n    overlap = train_patients & test_patients\n    print(f"\\nUnique patients -> train: {len(train_patients)}, test: {len(test_patients)}")\n    print(f"Patient overlap between train/test (Kaggle\'s own split): {len(overlap)}")\n\n    dup = train_idx.groupby("patient_id").size()\n    multi_image_patients = (dup > 1).sum()\n    print(f"Train patients with >1 image: {multi_image_patients} / {len(dup)}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/build_index.py")

wrote src/build_index.py


In [6]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_splits.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1b: Patient-level, multilabel-stratified K-fold split.\n\nWhy patient-level: build_index.py found 483/768 train patients have >1 image,\nso a plain random (or even plain multilabel-stratified) image-level split can\nput the same patient\'s images in both train and validation -> leaky, overly\noptimistic validation scores. We group by patient_id first, then stratify.\n\nWhy multilabel-stratified: UNIFESP\'s 22 classes are heavily imbalanced\n(Chest 738 vs Clavicles 9), so a plain GroupKFold can starve a fold of a rare\nclass entirely. We stratify on each patient\'s aggregated (union) label\nvector using iterative stratification.\n\nOutput: splits/fold_{i}.csv with columns [uid, fold] for every train image.\n"""\nimport os\nimport numpy as np\nimport pandas as pd\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nfrom . import config as C\n\nN_CLASSES = C.N_CLASSES\nN_FOLDS = 5\nSEED = C.SEED\n\n\ndef main():\n    index_df = pd.read_csv(os.path.join(C.INDEX_ROOT, "index.csv"))\n    train_df = index_df[index_df["split"] == "train"].copy()\n\n    label_cols = [c for c in train_df.columns if c.startswith("label_")]\n    assert len(label_cols) == N_CLASSES, label_cols\n\n    # aggregate to patient level: union of all labels this patient\'s images carry\n    patient_labels = (\n        train_df.groupby("patient_id")[label_cols]\n        .max()\n        .reset_index()\n    )\n    patients = patient_labels["patient_id"].values\n    Y = patient_labels[label_cols].values.astype(int)\n\n    print(f"Patients: {len(patients)} | images: {len(train_df)}")\n    print("Per-class positive patient count:\\n", pd.Series(Y.sum(axis=0), index=label_cols))\n\n    mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)\n    patient_fold = np.full(len(patients), -1, dtype=int)\n    for fold, (_, val_idx) in enumerate(mskf.split(patients, Y)):\n        patient_fold[val_idx] = fold\n\n    patient_to_fold = dict(zip(patients, patient_fold))\n    train_df["fold"] = train_df["patient_id"].map(patient_to_fold)\n\n    assert (train_df["fold"] >= 0).all(), "every image must get a fold"\n\n    out_cols = ["uid", "patient_id", "fold"] + label_cols\n    splits_dir = os.path.join(C.INDEX_ROOT, "splits")\n    os.makedirs(splits_dir, exist_ok=True)\n    out_path = os.path.join(splits_dir, "folds.csv")\n    train_df[out_cols].to_csv(out_path, index=False)\n    print("Saved", out_path)\n\n    # sanity: no patient split across folds, per-fold class coverage\n    leak = train_df.groupby("patient_id")["fold"].nunique()\n    print("Patients appearing in >1 fold (should be 0):", (leak > 1).sum())\n\n    print("\\nPer-fold image count:\\n", train_df["fold"].value_counts().sort_index())\n    print("\\nPer-fold positive image count per class:")\n    print(train_df.groupby("fold")[label_cols].sum())\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_splits.py")

wrote src/make_splits.py


In [7]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_holdout_cv_splits.py", "w", encoding="utf-8") as f:
    f.write('"""\nNotebook 2\'s split scheme: a permanent, never-trained-on held-out test set,\nplus a fresh 5-fold CV split restricted to the remaining pool.\n\nThis is different from make_splits.py\'s plain 5-fold split (where every\nfold is, in turn, somebody\'s validation set -- fine for model *selection*,\nbut every image still ends up influencing model checkpointing at some\npoint). Here, one of the original 5 folds (--holdout_fold, default 4) is\ncarved out as a fixed test set that no training run -- across any of the\n5 new CV folds -- ever sees, train or val. Each CV fold\'s best checkpoint\nis later evaluated against this same fixed set (see evaluate_on_holdout.py),\ngiving a genuine mean +/- std generalization estimate instead of 5 folds\nthat are each other\'s test set.\n\nPatient-level throughout (same reasoning as make_splits.py: many UNIFESP\npatients have >1 image, so splitting by image would leak).\n\nOutput:\n    <INDEX_ROOT>/splits/holdout_uids.csv   -- uid column, the fixed test set\n    <INDEX_ROOT>/splits/cv_folds.csv       -- uid, cv_fold (0..4), pool only\n\nUsage (after make_splits.py has already produced splits/folds.csv):\n    python -m src.make_holdout_cv_splits --holdout_fold 4\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nfrom . import config as C\n\nN_CLASSES = C.N_CLASSES\nN_CV_FOLDS = 5\nSEED = C.SEED\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--holdout_fold", type=int, default=4,\n                    help="which of the original 5 folds (from make_splits.py) becomes the fixed holdout test set")\n    args = p.parse_args()\n\n    folds_df = pd.read_csv(C.FOLDS_CSV)\n    label_cols = [c for c in folds_df.columns if c.startswith("label_")]\n    assert len(label_cols) == N_CLASSES, label_cols\n\n    holdout_df = folds_df[folds_df["fold"] == args.holdout_fold].reset_index(drop=True)\n    pool_df = folds_df[folds_df["fold"] != args.holdout_fold].reset_index(drop=True)\n    print(f"Holdout test (original fold {args.holdout_fold}): {len(holdout_df)} images, "\n          f"{holdout_df[\'patient_id\'].nunique()} patients")\n    print(f"CV pool (remaining images): {len(pool_df)} images, {pool_df[\'patient_id\'].nunique()} patients "\n          f"({len(pool_df) / len(folds_df):.1%} of all train images)")\n\n    # fresh patient-level multilabel-stratified 5-fold split, restricted to the pool only\n    patient_labels = pool_df.groupby("patient_id")[label_cols].max().reset_index()\n    patients = patient_labels["patient_id"].values\n    Y = patient_labels[label_cols].values.astype(int)\n\n    mskf = MultilabelStratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)\n    patient_cv_fold = np.full(len(patients), -1, dtype=int)\n    for cv_fold, (_, val_idx) in enumerate(mskf.split(patients, Y)):\n        patient_cv_fold[val_idx] = cv_fold\n    patient_to_cv_fold = dict(zip(patients, patient_cv_fold))\n    pool_df = pool_df.copy()\n    pool_df["cv_fold"] = pool_df["patient_id"].map(patient_to_cv_fold)\n    assert (pool_df["cv_fold"] >= 0).all()\n\n    # sanity: no patient split across folds or across the holdout boundary\n    leak = pool_df.groupby("patient_id")["cv_fold"].nunique()\n    assert (leak <= 1).all(), "a pool patient landed in >1 cv_fold"\n    assert set(holdout_df["patient_id"]) & set(pool_df["patient_id"]) == set(), \\\n        "a patient appears in both the holdout set and the CV pool"\n\n    print("\\nPer-cv_fold image count:\\n", pool_df["cv_fold"].value_counts().sort_index())\n    print("\\nPer-cv_fold positive image count per class:")\n    print(pool_df.groupby("cv_fold")[label_cols].sum())\n    print("\\nHoldout set positive image count per class:")\n    print(holdout_df[label_cols].sum())\n\n    splits_dir = os.path.join(C.INDEX_ROOT, "splits")\n    os.makedirs(splits_dir, exist_ok=True)\n\n    holdout_out = os.path.join(splits_dir, "holdout_uids.csv")\n    holdout_df[["uid", "patient_id"] + label_cols].to_csv(holdout_out, index=False)\n    print("\\nSaved:", holdout_out)\n\n    cv_out = os.path.join(splits_dir, "cv_folds.csv")\n    pool_df[["uid", "patient_id", "cv_fold"] + label_cols].rename(columns={"cv_fold": "fold"}).to_csv(\n        cv_out, index=False\n    )\n    print("Saved:", cv_out)\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_holdout_cv_splits.py")

wrote src/make_holdout_cv_splits.py


In [8]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/cache_images.py", "w", encoding="utf-8") as f:
    f.write('"""\nOptional one-time speed-up: pre-decode every DICOM into a PNG cache before\nlaunching the 8 parallel training runs. UnifespDataset already caches\nlazily on first read (see dataset.py\'s load_cached_or_decode), so this\nscript isn\'t required for correctness -- it just avoids every one of the\nfirst 2 (parallel) backbones redundantly decoding the same ~1700 training\nimages during their first epoch. Top Kaggle finishers on this exact\ncompetition reported DICOM decoding as their biggest bottleneck.\n\nKaggle usage, right after environment setup, before Step 2 training:\n    python -m src.cache_images\n"""\nimport time\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path, load_cached_or_decode\n\n\ndef main():\n    if C.CACHE_DIR is None:\n        print("UNIFESP_CACHE_DIR / CACHE_DIR is not set -- nothing to do. "\n              "Caching is off by default outside Kaggle; set the env var if you want it locally.")\n        return\n\n    ext = "PNG" if C.DATASET in ("chestxray14", "mura", "covidcxr") else "DICOM"\n    print(f"Indexing {ext} files under", C.DATA_ROOT, "...")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n    print(f"Found {len(uid_to_path)} files. Caching to {C.CACHE_DIR} ...")\n\n    t0 = time.time()\n    for i, (uid, path) in enumerate(uid_to_path.items()):\n        load_cached_or_decode(uid, path, C.CACHE_DIR)\n        if (i + 1) % 250 == 0:\n            print(f"  {i + 1}/{len(uid_to_path)} ({time.time() - t0:.1f}s elapsed)")\n\n    print(f"Done. Cached {len(uid_to_path)} images in {time.time() - t0:.1f}s -> {C.CACHE_DIR}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/cache_images.py")

wrote src/cache_images.py


In [9]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/evalkit.py", "w", encoding="utf-8") as f:
    f.write('"""\nevalkit -- leakage-free evaluation utilities for the UNIFESP body-part project.\n\nPure numpy (no torch), so it runs anywhere: Kaggle, laptop, CI.\n\nWhat it fixes compared with the earlier notebooks\n-------------------------------------------------\n1. Thresholds are tuned on OUT-OF-FOLD (OOF) predictions of the CV pool and then\n   FROZEN before they touch the held-out set (``tune_thresholds_oof`` +\n   ``nested_evaluate``).  The old ``ensemble_eval.py`` tuned thresholds on the very\n   fold it then scored, which inflated sample_mean_f1 (0.9356 / 0.9445).\n2. "At least one label per image" decoding (``predict_labels(min_one=True)``): ~92 %\n   of UNIFESP images are single-label and an empty prediction scores F1 = 0.\n3. Patient-level (cluster) bootstrap confidence intervals and a paired bootstrap\n   test for model-vs-model differences (``cluster_bootstrap``, ``paired_bootstrap``).\n4. Caruana greedy ensemble selection on OOF predictions (``caruana_ensemble``).\n5. Kaggle submission writer + validator (``write_submission``, ``check_submission``).\n\nMetric\n------\n``sample_mean_f1`` = per-image F1 between predicted and true label sets, averaged over\nimages.  This is the Kaggle "Mean F1-Score" of the UNIFESP competition.\n"""\nfrom __future__ import annotations\n\nimport csv\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\nN_CLASSES = 22\nDEFAULT_GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)\n\n\n# --------------------------------------------------------------------------- metrics\ndef per_image_f1(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:\n    """F1 per image. Both-empty -> 1.0, true-nonempty & pred-empty -> 0.0."""\n    y_true = np.asarray(y_true).astype(np.int64)\n    y_pred = np.asarray(y_pred).astype(np.int64)\n    tp = (y_true & y_pred).sum(1)\n    fp = ((1 - y_true) & y_pred).sum(1)\n    fn = (y_true & (1 - y_pred)).sum(1)\n    denom = 2 * tp + fp + fn\n    return np.where(denom == 0, 1.0, 2.0 * tp / np.maximum(denom, 1))\n\n\ndef sample_mean_f1(y_true, y_pred) -> float:\n    return float(per_image_f1(y_true, y_pred).mean())\n\n\ndef macro_f1(y_true, y_pred) -> float:\n    """Mean of per-class F1 (zero_division=0, same as sklearn\'s default for this use)."""\n    y_true = np.asarray(y_true).astype(np.int64)\n    y_pred = np.asarray(y_pred).astype(np.int64)\n    tp = (y_true & y_pred).sum(0)\n    fp = ((1 - y_true) & y_pred).sum(0)\n    fn = (y_true & (1 - y_pred)).sum(0)\n    denom = 2 * tp + fp + fn\n    f1 = np.where(denom == 0, 0.0, 2.0 * tp / np.maximum(denom, 1))\n    return float(f1.mean())\n\n\ndef micro_f1(y_true, y_pred) -> float:\n    y_true = np.asarray(y_true).astype(np.int64)\n    y_pred = np.asarray(y_pred).astype(np.int64)\n    tp = (y_true & y_pred).sum()\n    fp = ((1 - y_true) & y_pred).sum()\n    fn = (y_true & (1 - y_pred)).sum()\n    denom = 2 * tp + fp + fn\n    return float(0.0 if denom == 0 else 2.0 * tp / denom)\n\n\ndef subset_accuracy(y_true, y_pred) -> float:\n    return float((np.asarray(y_true) == np.asarray(y_pred)).all(axis=1).mean())\n\n\ndef all_metrics(y_true, y_pred) -> dict:\n    return {\n        "sample_mean_f1": sample_mean_f1(y_true, y_pred),\n        "macro_f1": macro_f1(y_true, y_pred),\n        "micro_f1": micro_f1(y_true, y_pred),\n        "subset_accuracy": subset_accuracy(y_true, y_pred),\n    }\n\n\n# --------------------------------------------------------------------------- decoding\ndef predict_labels(prob: np.ndarray, thresholds=0.5, min_one: bool = True,\n                   max_labels: int | None = None) -> np.ndarray:\n    """Probabilities -> 0/1 matrix.\n\n    min_one:    every image gets at least its best-scoring class (score = prob / threshold,\n                so a class with a low tuned threshold is not unfairly skipped).\n    max_labels: keep at most this many labels per image (by the same score).\n    """\n    prob = np.asarray(prob, dtype=np.float64)\n    thr = np.broadcast_to(np.asarray(thresholds, dtype=np.float64), (prob.shape[1],))\n    pred = (prob >= thr[None, :]).astype(np.int64)\n    score = prob / np.maximum(thr[None, :], 1e-6)\n    if min_one:\n        empty = pred.sum(1) == 0\n        if empty.any():\n            best = score[empty].argmax(1)\n            pred[np.where(empty)[0], best] = 1\n    if max_labels is not None:\n        order = np.argsort(-score, axis=1)\n        keep = np.zeros_like(pred)\n        rows = np.arange(prob.shape[0])[:, None]\n        keep[rows, order[:, :max_labels]] = 1\n        pred = pred * keep\n        if min_one:  # top-1 is always kept, but be explicit\n            pred[np.arange(prob.shape[0]), order[:, 0]] = 1\n    return pred\n\n\n# --------------------------------------------------------------------------- thresholds\ndef tune_thresholds_oof(oof_prob: np.ndarray, y: np.ndarray, grid: Sequence[float] = DEFAULT_GRID,\n                        base: float = 0.5, n_rounds: int = 3, min_pos: float = 20.0,\n                        min_one: bool = True) -> np.ndarray:\n    """Per-class thresholds by coordinate ascent on the REAL objective (sample_mean_f1).\n\n    * Use ONLY out-of-fold predictions of the CV pool here -- never the held-out set and\n      never the same fold you later report on.\n    * Rare classes are shrunk toward ``base`` with weight n_pos / (n_pos + min_pos), because\n      a threshold fitted on 5-10 positives is mostly noise.\n    """\n    oof_prob = np.asarray(oof_prob, dtype=np.float64)\n    y = np.asarray(y).astype(np.int64)\n    C = oof_prob.shape[1]\n    thr = np.full(C, base, dtype=np.float64)\n\n    def score(t):\n        return sample_mean_f1(y, predict_labels(oof_prob, t, min_one=min_one))\n\n    best = score(thr)\n    for _ in range(n_rounds):\n        changed = False\n        for c in range(C):\n            cur = thr[c]\n            best_t, best_s = cur, best\n            for g in grid:\n                if g == cur:\n                    continue\n                thr[c] = g\n                s = score(thr)\n                # strictly better, or equal but closer to the base threshold (prefer simple)\n                if s > best_s + 1e-12 or (abs(s - best_s) <= 1e-12 and abs(g - base) < abs(best_t - base)):\n                    best_t, best_s = g, s\n            thr[c] = best_t\n            if best_t != cur:\n                changed = True\n            best = best_s\n        if not changed:\n            break\n    n_pos = y.sum(0).astype(np.float64)\n    lam = n_pos / (n_pos + min_pos)\n    return lam * thr + (1.0 - lam) * base\n\n\ndef nested_evaluate(oof_prob, y_oof, hold_prob, y_hold, **tune_kw) -> dict:\n    """The only honest way to report tuned-threshold numbers.\n\n    thresholds <- OOF of the CV pool;  scores <- held-out set (never used for any choice).\n    Returns the metrics for three decoding rules so each lever\'s gain is visible.\n    """\n    thr = tune_thresholds_oof(oof_prob, y_oof, **tune_kw)\n    out = {\n        "thresholds": thr,\n        "hold_0.5": all_metrics(y_hold, predict_labels(hold_prob, 0.5, min_one=False)),\n        "hold_0.5_min_one": all_metrics(y_hold, predict_labels(hold_prob, 0.5, min_one=True)),\n        "hold_tuned_min_one": all_metrics(y_hold, predict_labels(hold_prob, thr, min_one=True)),\n    }\n    return out\n\n\n# --------------------------------------------------------------------------- bootstrap\ndef _cluster_indices(groups: np.ndarray):\n    groups = np.asarray(groups)\n    uniq, inv = np.unique(groups, return_inverse=True)\n    members = [np.where(inv == k)[0] for k in range(len(uniq))]\n    return uniq, members\n\n\ndef cluster_bootstrap(metric_fn, y_true, y_pred, groups, n_boot: int = 2000, seed: int = 0,\n                      alpha: float = 0.05) -> dict:\n    """Patient-level bootstrap: resample PATIENTS (all their images together), recompute metric.\n\n    metric_fn(y_true, y_pred) -> float.  ``groups`` = patient id per image.\n    """\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    _, members = _cluster_indices(groups)\n    P = len(members)\n    rng = np.random.default_rng(seed)\n    vals = np.empty(n_boot)\n    for b in range(n_boot):\n        pick = rng.integers(0, P, size=P)\n        idx = np.concatenate([members[k] for k in pick])\n        vals[b] = metric_fn(y_true[idx], y_pred[idx])\n    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])\n    return {"estimate": float(metric_fn(y_true, y_pred)), "ci_low": float(lo), "ci_high": float(hi),\n            "boot_std": float(vals.std(ddof=1)), "n_patients": P, "n_boot": n_boot}\n\n\ndef paired_bootstrap(metric_fn, y_true, pred_a, pred_b, groups, n_boot: int = 2000, seed: int = 0,\n                     alpha: float = 0.05) -> dict:\n    """Is model A better than B?  Same patient resamples for both; returns CI of (A - B)."""\n    y_true = np.asarray(y_true)\n    pred_a = np.asarray(pred_a)\n    pred_b = np.asarray(pred_b)\n    _, members = _cluster_indices(groups)\n    P = len(members)\n    rng = np.random.default_rng(seed)\n    d = np.empty(n_boot)\n    for b in range(n_boot):\n        pick = rng.integers(0, P, size=P)\n        idx = np.concatenate([members[k] for k in pick])\n        d[b] = metric_fn(y_true[idx], pred_a[idx]) - metric_fn(y_true[idx], pred_b[idx])\n    lo, hi = np.quantile(d, [alpha / 2, 1 - alpha / 2])\n    # two-sided bootstrap p-value for H0: difference == 0\n    p = 2.0 * min((d <= 0).mean(), (d >= 0).mean())\n    return {"diff": float(metric_fn(y_true, pred_a) - metric_fn(y_true, pred_b)),\n            "ci_low": float(lo), "ci_high": float(hi), "p_value": float(min(1.0, p)), "n_boot": n_boot}\n\n\n# --------------------------------------------------------------------------- ensembling\ndef average_probs(prob_list: Iterable[np.ndarray], weights=None, mode: str = "prob") -> np.ndarray:\n    arr = np.stack([np.asarray(p, dtype=np.float64) for p in prob_list])\n    w = np.ones(len(arr)) if weights is None else np.asarray(weights, dtype=np.float64)\n    w = w / w.sum()\n    if mode == "logit":\n        eps = 1e-6\n        lg = np.log(np.clip(arr, eps, 1 - eps) / (1 - np.clip(arr, eps, 1 - eps)))\n        z = np.tensordot(w, lg, axes=1)\n        return 1.0 / (1.0 + np.exp(-z))\n    return np.tensordot(w, arr, axes=1)\n\n\ndef caruana_ensemble(oof_list: Sequence[np.ndarray], y: np.ndarray, n_iter: int = 40,\n                     init_best: bool = True) -> np.ndarray:\n    """Greedy forward selection WITH replacement on OOF sample_mean_f1 (0.5 thr, min_one).\n\n    Returns weights (sum 1).  Fit on OOF only; apply the weights unchanged to held-out/test.\n    """\n    k = len(oof_list)\n    counts = np.zeros(k)\n\n    def sc(w):\n        p = average_probs(oof_list, w)\n        return sample_mean_f1(y, predict_labels(p, 0.5, min_one=True))\n\n    if init_best:\n        singles = [sc(np.eye(k)[i]) for i in range(k)]\n        counts[int(np.argmax(singles))] = 1\n    for _ in range(n_iter):\n        best_i, best_s = None, -1.0\n        for i in range(k):\n            trial = counts.copy()\n            trial[i] += 1\n            s = sc(trial)\n            if s > best_s:\n                best_i, best_s = i, s\n        counts[best_i] += 1\n    return counts / counts.sum()\n\n\ndef combine_heads(p_sigmoid: np.ndarray, p_softmax: np.ndarray, alpha: float = 0.5) -> np.ndarray:\n    """Blend a multilabel (sigmoid) head with a single-label (softmax) head."""\n    return alpha * np.asarray(p_sigmoid) + (1.0 - alpha) * np.asarray(p_softmax)\n\n\n# --------------------------------------------------------------------------- submission\ndef write_submission(uids: Sequence[str], prob: np.ndarray, thresholds, path: str,\n                     min_one: bool = True, max_labels: int | None = None) -> np.ndarray:\n    pred = predict_labels(prob, thresholds, min_one=min_one, max_labels=max_labels)\n    with open(path, "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["SOPInstanceUID", "Target"])\n        for u, row in zip(uids, pred):\n            w.writerow([u, " ".join(str(int(c)) for c in np.where(row == 1)[0])])\n    return pred\n\n\ndef check_submission(path: str, sample_path: str, n_classes: int = N_CLASSES) -> dict:\n    """Validate a submission against Kaggle\'s sample_submission.csv. Raises AssertionError."""\n    with open(path, newline="") as f:\n        sub = list(csv.DictReader(f))\n    with open(sample_path, newline="") as f:\n        ref = list(csv.DictReader(f))\n    assert list(sub[0].keys()) == ["SOPInstanceUID", "Target"], f"bad header {list(sub[0].keys())}"\n    assert len(sub) == len(ref), f"row count {len(sub)} != {len(ref)}"\n    assert {r["SOPInstanceUID"] for r in sub} == {r["SOPInstanceUID"] for r in ref}, "id set differs"\n    assert len({r["SOPInstanceUID"] for r in sub}) == len(sub), "duplicate ids"\n    n_labels = []\n    for r in sub:\n        labs = r["Target"].split()\n        assert len(labs) >= 1, f"empty Target for {r[\'SOPInstanceUID\']}"\n        ints = [int(x) for x in labs]\n        assert all(0 <= i < n_classes for i in ints), f"label out of range: {labs}"\n        assert len(set(ints)) == len(ints), f"duplicate labels: {labs}"\n        n_labels.append(len(ints))\n    return {"rows": len(sub), "mean_labels_per_image": float(np.mean(n_labels)),\n            "frac_single_label": float(np.mean(np.array(n_labels) == 1))}\n')
print("wrote src/evalkit.py")

wrote src/evalkit.py


In [10]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/train_v2.py", "w", encoding="utf-8") as f:
    f.write('"""\ntrain_v2 -- one training run (one backbone, one fold) designed for CLEAN out-of-fold (OOF)\npredictions.\n\nDifferences from src/train.py / engine.py (and why)\n---------------------------------------------------\n* NO checkpoint selection.  The old engine kept the epoch with the best validation\n  macro_f1 of the SAME fold that was later reported / used to tune thresholds, which makes\n  every reported number optimistic.  Here the schedule is fixed (cosine, N epochs) and the\n  final EMA weights are used.  Per-epoch validation numbers are printed for monitoring only.\n* EMA of the weights (smoother, usually better than the last raw epoch, needs no selection).\n* Two heads on one backbone: a multilabel sigmoid head (22 logits) and a single-label\n  softmax head (22 logits).  ~92 % of UNIFESP images carry exactly one label, and the\n  2nd-place Kaggle solution framed it as single-label multiclass.  Both heads are saved;\n  evalkit/run_ensemble choose / blend them on OOF only.\n* Predictions for OOF fold, held-out set (optional) and the Kaggle test set are written to\n  ``preds.npz`` so the ensemble stage is pure numpy (no GPU).\n* fp16 autocast with logits-space losses only (the V3 crash was BCE on probabilities).\n\nUsage (Kaggle, GPU):\n    python -m src.train_v2 --backbone swin_tiny --img_size 224 --fold 0 \\\n        --folds_csv /kaggle/working/splits/cv_folds.csv --fold_col fold \\\n        --holdout_csv /kaggle/working/splits/holdout_uids.csv --run_name swin_tiny_s224_f0\n\nSmoke test (CPU, a few images, no pretrained weights, no real training):\n    python -m src.train_v2 --backbone resnet18 --img_size 64 --fold 0 --epochs 1 --smoke_test ...\n"""\nimport argparse\nimport copy\nimport json\nimport os\nimport random\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom . import config as C\nfrom . import evalkit as E\nfrom .dataset import UnifespDataset, build_uid_to_path\n\nN = C.N_CLASSES\nLABEL_COLS = [f"label_{k}_{C.LABEL_NAMES[k].replace(\' \', \'_\')}" for k in range(N)]\n\n# short name -> timm name (anything else is passed to timm unchanged)\nTIMM_NAMES = {\n    "swin_tiny": "swin_tiny_patch4_window7_224",\n    "efficientnetv2_s": "tf_efficientnetv2_s",\n    "densenet121": "densenet121",\n    "convnext_tiny": "convnext_tiny",\n    "convnext_small": "convnext_small",\n    "resnet50": "resnet50",\n    "resnet152": "resnet152",\n    "efficientnet_b3": "efficientnet_b3",\n}\n\n\n# ----------------------------------------------------------------------------- model\nclass TwoHeadNet(nn.Module):\n    """timm backbone (pooled features) + one linear layer producing 2*N logits:\n    [:N] -> multilabel (sigmoid), [N:] -> single-label (softmax)."""\n\n    def __init__(self, backbone, pretrained=True, drop=0.2):\n        super().__init__()\n        import timm\n        name = TIMM_NAMES.get(backbone, backbone)\n        self.body = timm.create_model(name, pretrained=pretrained, num_classes=0)\n        self.drop = nn.Dropout(drop)\n        self.head = nn.Linear(self.body.num_features, 2 * N)\n\n    def forward(self, x):\n        z = self.head(self.drop(self.body(x)))\n        return z[:, :N], z[:, N:]\n\n\ndef compute_pos_weight(y, cap=20.0):\n    pos = np.clip(y.sum(0), 1, None)\n    return torch.tensor(np.clip((len(y) - pos) / pos, 1.0, cap), dtype=torch.float32)\n\n\nclass TwoHeadLoss(nn.Module):\n    """BCE-with-logits (pos_weight, capped) on the sigmoid head\n       + lam * soft-target cross-entropy on the softmax head (target = y / y.sum())."""\n\n    def __init__(self, pos_weight=None, lam=0.5, smoothing=0.0):\n        super().__init__()\n        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n        self.lam, self.smoothing = lam, smoothing\n\n    def forward(self, outs, y):\n        sig, soft = outs\n        loss = self.bce(sig.float(), y)\n        if self.lam > 0:\n            t = y / y.sum(1, keepdim=True).clamp(min=1.0)\n            if self.smoothing > 0:\n                t = t * (1 - self.smoothing) + self.smoothing / N\n            loss = loss + self.lam * (-(t * F.log_softmax(soft.float(), dim=1)).sum(1)).mean()\n        return loss\n\n\nclass EMA:\n    """Exponential moving average of parameters AND buffers (BN statistics)."""\n\n    def __init__(self, model, decay=0.998):\n        self.ema = copy.deepcopy(model).eval()\n        for p in self.ema.parameters():\n            p.requires_grad_(False)\n        self.decay, self.steps = decay, 0\n\n    @torch.no_grad()\n    def update(self, model):\n        self.steps += 1\n        d = min(self.decay, (1 + self.steps) / (10 + self.steps))\n        msd = model.state_dict()\n        for k, v in self.ema.state_dict().items():\n            if v.dtype.is_floating_point:\n                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)\n            else:\n                v.copy_(msd[k])\n\n\n# ----------------------------------------------------------------------------- data\ndef get_transforms(img_size, train, aug="light"):\n    import albumentations as A\n    from albumentations.pytorch import ToTensorV2\n    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)\n    if not train:\n        return A.Compose([A.Resize(img_size, img_size), A.Normalize(mean=mean, std=std), ToTensorV2()])\n    if aug == "heavy":      # in the spirit of the 2nd-place Kaggle writeup; defaults only (API-stable)\n        ops = [A.Resize(img_size, img_size), A.HorizontalFlip(p=0.5),\n               A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=15, p=0.6),\n               A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),\n               A.GaussNoise(p=0.2), A.GaussianBlur(p=0.1)]\n    else:                   # "light": the project\'s earlier default\n        ops = [A.Resize(img_size, img_size), A.HorizontalFlip(p=0.5),\n               A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),\n               A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5)]\n    return A.Compose(ops + [A.Normalize(mean=mean, std=std), ToTensorV2()])\n\n\ndef make_frames(args):\n    index_df = pd.read_csv(C.INDEX_CSV)\n    folds = pd.read_csv(args.folds_csv)[["uid", args.fold_col]]\n    tr_all = index_df[index_df["split"] == "train"].merge(folds, on="uid", how="inner")\n    train_df = tr_all[tr_all[args.fold_col] != args.fold].reset_index(drop=True)\n    oof_df = tr_all[tr_all[args.fold_col] == args.fold].reset_index(drop=True)\n    hold_df = None\n    if args.holdout_csv:\n        hold_uids = set(pd.read_csv(args.holdout_csv)["uid"])\n        hold_df = index_df[index_df["uid"].isin(hold_uids)].reset_index(drop=True)\n    test_df = index_df[index_df["split"] == "test"].reset_index(drop=True)\n    if args.smoke_test:\n        train_df, oof_df, test_df = train_df.iloc[:8], oof_df.iloc[:4], test_df.iloc[:4]\n        hold_df = None if hold_df is None else hold_df.iloc[:4]\n    return train_df, oof_df, hold_df, test_df\n\n\n# ----------------------------------------------------------------------------- predict\n@torch.no_grad()\ndef predict(model, loader, device, tta=False):\n    model.eval()\n    sig_all, soft_all, uids = [], [], []\n    for images, second in loader:\n        images = images.to(device, non_blocking=True)\n        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):\n            s, f = model(images)\n            ps, pf = torch.sigmoid(s.float()), torch.softmax(f.float(), dim=1)\n            if tta:\n                s2, f2 = model(torch.flip(images, dims=[3]))\n                ps = (ps + torch.sigmoid(s2.float())) / 2\n                pf = (pf + torch.softmax(f2.float(), dim=1)) / 2\n        sig_all.append(ps.cpu().numpy()); soft_all.append(pf.cpu().numpy())\n        uids.extend(list(second) if not torch.is_tensor(second) else [None] * len(images))\n    return np.concatenate(sig_all).astype(np.float32), np.concatenate(soft_all).astype(np.float32), uids\n\n\ndef run_predictions(model, df, has_labels, args, device, uid_to_path, img_size):\n    ds = UnifespDataset(df, C.DATA_ROOT, transform=get_transforms(img_size, False), has_labels=False,\n                        uid_to_path=uid_to_path)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)\n    out = {}\n    for tag, tta in (("", False), ("_tta", True)):\n        s, f, uids = predict(model, dl, device, tta=tta)\n        out["sig" + tag], out["soft" + tag] = s, f\n    out["uid"] = np.array(uids if uids and uids[0] is not None else df["uid"].tolist())\n    if has_labels:\n        out["y"] = df[LABEL_COLS].to_numpy(dtype=np.int64)\n    return out\n\n\n# ----------------------------------------------------------------------------- main\ndef set_seed(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--backbone", required=True)\n    p.add_argument("--img_size", type=int, default=224)\n    p.add_argument("--fold", type=int, required=True)\n    p.add_argument("--folds_csv", required=True)\n    p.add_argument("--fold_col", default="fold")\n    p.add_argument("--holdout_csv", default=None)\n    p.add_argument("--epochs", type=int, default=30)\n    p.add_argument("--batch_size", type=int, default=32)\n    p.add_argument("--lr", type=float, default=3e-4)\n    p.add_argument("--wd", type=float, default=1e-4)\n    p.add_argument("--warmup_epochs", type=float, default=1.0)\n    p.add_argument("--ema", type=float, default=0.998)\n    p.add_argument("--aug", default="light", choices=["light", "heavy"])\n    p.add_argument("--lam_ce", type=float, default=0.5, help="weight of the softmax single-label head loss (0 = off)")\n    p.add_argument("--smoothing", type=float, default=0.0)\n    p.add_argument("--seed", type=int, default=C.SEED)\n    p.add_argument("--num_workers", type=int, default=4)\n    p.add_argument("--run_name", default=None)\n    p.add_argument("--out_dir", default=C.OUTPUT_DIR)\n    p.add_argument("--smoke_test", action="store_true", help="tiny subset, no pretrained weights (CPU code-path check)")\n    p.add_argument("--no_pretrained", action="store_true", help="random init (local sanity runs only)")\n    args = p.parse_args()\n\n    set_seed(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    run_name = args.run_name or f"{args.backbone}_s{args.img_size}_f{args.fold}"\n    run_dir = os.path.join(args.out_dir, run_name)\n    os.makedirs(run_dir, exist_ok=True)\n\n    train_df, oof_df, hold_df, test_df = make_frames(args)\n    print(f"[{run_name}] train={len(train_df)} oof={len(oof_df)} "\n          f"holdout={0 if hold_df is None else len(hold_df)} test={len(test_df)} device={device}")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n\n    tr_ds = UnifespDataset(train_df, C.DATA_ROOT, transform=get_transforms(args.img_size, True, args.aug),\n                           uid_to_path=uid_to_path)\n    va_ds = UnifespDataset(oof_df, C.DATA_ROOT, transform=get_transforms(args.img_size, False),\n                           uid_to_path=uid_to_path)\n    tr_dl = DataLoader(tr_ds, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers,\n                       pin_memory=device.type == "cuda", drop_last=len(train_df) > args.batch_size)\n    va_dl = DataLoader(va_ds, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)\n\n    model = TwoHeadNet(args.backbone, pretrained=not (args.smoke_test or args.no_pretrained)).to(device)\n    ema = EMA(model, args.ema)\n    loss_fn = TwoHeadLoss(compute_pos_weight(train_df[LABEL_COLS].to_numpy(np.float32)).to(device),\n                          lam=args.lam_ce, smoothing=args.smoothing)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)\n    steps_per_epoch = max(1, len(tr_dl))\n    total, warm = args.epochs * steps_per_epoch, int(args.warmup_epochs * steps_per_epoch)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: (s + 1) / max(1, warm) if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm))))\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n\n    t0, history = time.time(), []\n    for epoch in range(args.epochs):\n        model.train(); run_loss = 0.0\n        for images, y in tr_dl:\n            images, y = images.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):\n                outs = model(images)\n            loss = loss_fn(outs, y)\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)\n            run_loss += float(loss.detach())\n        # monitoring only -- NEVER used to pick a checkpoint\n        s, _, _ = predict(ema.ema, va_dl, device)\n        yv = oof_df[LABEL_COLS].to_numpy(np.int64)\n        mon = E.sample_mean_f1(yv, E.predict_labels(s, 0.5, min_one=True))\n        history.append({"epoch": epoch, "train_loss": run_loss / steps_per_epoch, "oof_sample_mean_f1_monitor": mon})\n        print(f"epoch {epoch}: loss={run_loss / steps_per_epoch:.4f} monitor_oof_f1={mon:.4f} ({time.time() - t0:.0f}s)")\n\n    # final EMA weights -> predictions (no selection)\n    preds = {}\n    for tag, df, has_y in (("oof", oof_df, True), ("hold", hold_df, True), ("test", test_df, False)):\n        if df is None:\n            continue\n        r = run_predictions(ema.ema, df, has_y, args, device, uid_to_path, args.img_size)\n        for k, v in r.items():\n            preds[f"{tag}_{k}"] = v\n    np.savez_compressed(os.path.join(run_dir, "preds.npz"), **preds)\n    torch.save(ema.ema.state_dict(), os.path.join(run_dir, "final_ema.pt"))   # state_dict only (weights_only-safe)\n    meta = {"args": vars(args), "history": history, "seconds": time.time() - t0, "torch": torch.__version__}\n    json.dump(meta, open(os.path.join(run_dir, "meta.json"), "w"), indent=1, default=str)\n    print(f"[{run_name}] done in {time.time() - t0:.0f}s -> {run_dir}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/train_v2.py")

wrote src/train_v2.py


In [11]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/run_ensemble.py", "w", encoding="utf-8") as f:
    f.write('"""\nrun_ensemble -- CPU-only stage that turns the per-fold ``preds.npz`` files written by\ntrain_v2 into (a) honest numbers and (b) a Kaggle submission.\n\nHonesty rules enforced here\n---------------------------\n* Every DECISION (which head / TTA variant per run, ensemble weights, per-class thresholds)\n  is made on the out-of-fold (OOF) predictions of the CV pool ONLY.\n* The held-out set is touched exactly once, at the end, to REPORT.\n* Held-out and Kaggle-test probabilities of a run are the average of its 5 fold models\n  (bagging), which is how the Kaggle submission is produced too.\n\nInputs:  <out_dir>/<run>_f<k>/preds.npz  for every run prefix and fold k.\nOutputs: <result_dir>/results.json, per_run.csv, ensemble_arrays.npz, submission.csv\n"""\nimport argparse\nimport json\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import evalkit as E\n\nN = 22\n\n\ndef _load_run(out_dir, run, folds, pool_uids, with_hold, with_test):\n    pos = {u: i for i, u in enumerate(pool_uids)}\n    keys = ["sig", "soft", "sig_tta", "soft_tta"]\n    oof = {k: np.full((len(pool_uids), N), np.nan, np.float32) for k in keys}\n    hold = {k: [] for k in keys}\n    test = {k: [] for k in keys}\n    hold_uid = test_uid = None\n    for f in folds:\n        d = np.load(os.path.join(out_dir, f"{run}_f{f}", "preds.npz"), allow_pickle=False)\n        idx = np.array([pos[u] for u in d["oof_uid"]])\n        for k in keys:\n            oof[k][idx] = d[f"oof_{k}"]\n        if with_hold and "hold_uid" in d.files:\n            hu = d["hold_uid"]\n            hold_uid = hu if hold_uid is None else hold_uid\n            assert (hu == hold_uid).all(), "holdout uid order differs between folds"\n            for k in keys:\n                hold[k].append(d[f"hold_{k}"])\n        if with_test and "test_uid" in d.files:\n            tu = d["test_uid"]\n            test_uid = tu if test_uid is None else test_uid\n            assert (tu == test_uid).all(), "test uid order differs between folds"\n            for k in keys:\n                test[k].append(d[f"test_{k}"])\n    for k in keys:\n        assert not np.isnan(oof[k]).any(), f"{run}: OOF not complete for {k} (folds={folds})"\n    avg = lambda L: {k: np.mean(L[k], axis=0) for k in keys} if L[keys[0]] else None\n    return {"oof": oof, "hold": avg(hold), "test": avg(test), "hold_uid": hold_uid, "test_uid": test_uid}\n\n\ndef _variants(p):\n    """Candidate probability matrices for one run (all selected on OOF later)."""\n    v = {"sig": p["sig"], "soft": p["soft"], "blend": E.combine_heads(p["sig"], p["soft"], 0.5),\n         "sig_tta": p["sig_tta"], "soft_tta": p["soft_tta"],\n         "blend_tta": E.combine_heads(p["sig_tta"], p["soft_tta"], 0.5)}\n    return v\n\n\ndef run(args):\n    os.makedirs(args.result_dir, exist_ok=True)\n    index = pd.read_csv(args.index_csv)\n    label_cols = [c for c in index.columns if c.startswith("label_")]\n    assert len(label_cols) == N\n    pool = pd.read_csv(args.cv_folds_csv)\n    pool_uids = pool["uid"].tolist()\n    lab = index.set_index("uid")\n    y_pool = lab.loc[pool_uids, label_cols].to_numpy(np.int64)\n\n    have_hold = bool(args.holdout_csv)\n    hold_uids = pd.read_csv(args.holdout_csv)["uid"].tolist() if have_hold else []\n    folds = [int(x) for x in args.folds.split(",")]\n    runs = [r for r in args.runs.split(",") if r]\n\n    data, chosen, rows = {}, {}, []\n    for r in runs:\n        d = _load_run(args.out_dir, r, folds, pool_uids, have_hold, True)\n        vs_oof = _variants(d["oof"])\n        scores = {k: E.sample_mean_f1(y_pool, E.predict_labels(v, 0.5, min_one=True)) for k, v in vs_oof.items()}\n        best = max(scores, key=scores.get)                     # decision on OOF only\n        chosen[r] = best\n        data[r] = {"oof": vs_oof[best],\n                   "hold": _variants(d["hold"])[best] if d["hold"] is not None else None,\n                   "test": _variants(d["test"])[best] if d["test"] is not None else None,\n                   "hold_uid": d["hold_uid"], "test_uid": d["test_uid"]}\n        rows.append({"run": r, "chosen_variant": best, **{f"oof_{k}": s for k, s in scores.items()}})\n\n    # ---- ensemble weights on OOF only\n    oof_list = [data[r]["oof"] for r in runs]\n    w = E.caruana_ensemble(oof_list, y_pool, n_iter=args.caruana_iters)\n    ens_oof = E.average_probs(oof_list, w)\n    thr = E.tune_thresholds_oof(ens_oof, y_pool, min_pos=args.min_pos)\n\n    result = {"runs": runs, "chosen_variant": chosen, "weights": dict(zip(runs, map(float, w))),\n              "thresholds": [float(t) for t in thr],\n              "oof": {"ens_0.5_min_one": E.all_metrics(y_pool, E.predict_labels(ens_oof, 0.5, True)),\n                      "note": "OOF numbers are for model selection only; report the held-out numbers"}}\n\n    # ---- held-out set: used once, to report\n    if have_hold:\n        hold_uid_order = data[runs[0]]["hold_uid"].tolist()\n        y_hold = lab.loc[hold_uid_order, label_cols].to_numpy(np.int64)\n        groups = lab.loc[hold_uid_order, "patient_id"].astype(str).to_numpy()\n        hold_list = [data[r]["hold"] for r in runs]\n        ens_hold = E.average_probs(hold_list, w)\n        pred_05 = E.predict_labels(ens_hold, 0.5, min_one=False)\n        pred_05m = E.predict_labels(ens_hold, 0.5, min_one=True)\n        pred_tuned = E.predict_labels(ens_hold, thr, min_one=True)\n        result["holdout"] = {"n_images": len(y_hold), "n_patients": int(len(set(groups))),\n                             "ens_0.5": E.all_metrics(y_hold, pred_05),\n                             "ens_0.5_min_one": E.all_metrics(y_hold, pred_05m),\n                             "ens_tuned_min_one": E.all_metrics(y_hold, pred_tuned)}\n        result["holdout"]["ci_sample_mean_f1_tuned"] = E.cluster_bootstrap(\n            E.sample_mean_f1, y_hold, pred_tuned, groups, n_boot=args.n_boot)\n        result["holdout"]["ci_macro_f1_tuned"] = E.cluster_bootstrap(\n            E.macro_f1, y_hold, pred_tuned, groups, n_boot=args.n_boot)\n        # best single run (chosen on OOF!) vs ensemble, paired on the same patient resamples\n        single_oof = {r: E.sample_mean_f1(y_pool, E.predict_labels(data[r]["oof"], 0.5, True)) for r in runs}\n        best_single = max(single_oof, key=single_oof.get)\n        pred_single = E.predict_labels(data[best_single]["hold"], 0.5, min_one=True)\n        result["holdout"]["best_single_by_oof"] = best_single\n        result["holdout"]["best_single_metrics"] = E.all_metrics(y_hold, pred_single)\n        result["holdout"]["paired_ens_vs_single_sample_f1"] = E.paired_bootstrap(\n            E.sample_mean_f1, y_hold, pred_tuned, pred_single, groups, n_boot=args.n_boot)\n        for row in rows:\n            r = row["run"]\n            row["holdout_sample_mean_f1_0.5_min_one"] = E.sample_mean_f1(\n                y_hold, E.predict_labels(data[r]["hold"], 0.5, min_one=True))\n\n    # ---- Kaggle test submission (average of the fold models, same weights / thresholds)\n    test_uid_order = data[runs[0]]["test_uid"].tolist()\n    ens_test = E.average_probs([data[r]["test"] for r in runs], w)\n    sub_path = os.path.join(args.result_dir, "submission.csv")\n    pred = E.write_submission(test_uid_order, ens_test, thr, sub_path, min_one=True)\n    result["submission"] = {"path": sub_path, "mean_labels": float(pred.sum(1).mean())}\n    if args.sample_submission:\n        result["submission"]["check"] = E.check_submission(sub_path, args.sample_submission)\n\n    pd.DataFrame(rows).to_csv(os.path.join(args.result_dir, "per_run.csv"), index=False)\n    json.dump(result, open(os.path.join(args.result_dir, "results.json"), "w"), indent=1, default=float)\n    np.savez_compressed(os.path.join(args.result_dir, "ensemble_arrays.npz"), oof=ens_oof,\n                        hold=ens_hold if have_hold else np.zeros(0), test=ens_test, thresholds=thr, weights=w)\n    return result\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--out_dir", required=True, help="folder that contains <run>_f<k>/preds.npz")\n    p.add_argument("--runs", required=True, help="comma-separated run prefixes, e.g. swin_tiny_s224,densenet121_s256")\n    p.add_argument("--folds", default="0,1,2,3,4")\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--cv_folds_csv", required=True)\n    p.add_argument("--holdout_csv", default=None)\n    p.add_argument("--sample_submission", default=None)\n    p.add_argument("--result_dir", required=True)\n    p.add_argument("--n_boot", type=int, default=2000)\n    p.add_argument("--caruana_iters", type=int, default=40)\n    p.add_argument("--min_pos", type=float, default=20.0)\n    res = run(p.parse_args())\n    print(json.dumps({k: res[k] for k in ("chosen_variant", "weights", "holdout") if k in res}, indent=1, default=float))\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/run_ensemble.py")

wrote src/run_ensemble.py


In [12]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/merge_runs.py", "w", encoding="utf-8") as f:
    f.write('"""\nmerge_runs -- helpers for running the pipeline as several notebooks (same Kaggle account) and merging.\n\nWhy a hash check: every part notebook rebuilds index.csv / splits from the same code and seed, so\nthey SHOULD be identical.  If they are not (different dataset version, library version, ...) the\nOOF predictions of the parts refer to different folds and must not be merged.\n\n    python -m src.merge_runs --write_hashes --index_csv index.csv --splits_dir splits --out split_hashes.json\n    python -m src.merge_runs --input_root /kaggle/input --out_dir runs --index_csv index.csv --splits_dir splits\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\n\nRUN_DIR = re.compile(r"^(?P<prefix>.+)_f(?P<fold>\\d+)$")\nHASHED = ("cv_folds.csv", "holdout_uids.csv", "folds.csv")\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef split_hashes(index_csv, splits_dir):\n    out = {"index.csv": sha256(index_csv)}\n    for n in HASHED:\n        p = os.path.join(splits_dir, n)\n        if os.path.exists(p):\n            out[n] = sha256(p)\n    return out\n\n\ndef discover(input_root):\n    """{prefix: {fold: dir}} for every <prefix>_f<k>/preds.npz found anywhere under input_root."""\n    found = {}\n    for root, dirs, files in os.walk(input_root):\n        if "preds.npz" in files:\n            m = RUN_DIR.match(os.path.basename(root))\n            if m:\n                found.setdefault(m["prefix"], {}).setdefault(int(m["fold"]), root)\n    return found\n\n\ndef check_hashes(input_root, local):\n    """Compare local split hashes with every split_hashes.json found under input_root."""\n    problems, seen = [], 0\n    for root, _d, files in os.walk(input_root):\n        if "split_hashes.json" in files:\n            seen += 1\n            theirs = json.load(open(os.path.join(root, "split_hashes.json")))\n            for k, v in theirs.items():\n                if k in local and local[k] != v:\n                    problems.append(f"{root}: {k} differs")\n    return seen, problems\n\n\ndef collect(input_root, out_dir, folds=(0, 1, 2, 3, 4)):\n    os.makedirs(out_dir, exist_ok=True)\n    found = discover(input_root)\n    complete, incomplete = [], {}\n    for prefix, by_fold in sorted(found.items()):\n        missing = [f for f in folds if f not in by_fold]\n        if missing:\n            incomplete[prefix] = missing\n            continue\n        for f, src in by_fold.items():\n            dst = os.path.join(out_dir, f"{prefix}_f{f}")\n            os.makedirs(dst, exist_ok=True)\n            for name in ("preds.npz", "meta.json"):\n                if os.path.exists(os.path.join(src, name)):\n                    shutil.copy2(os.path.join(src, name), os.path.join(dst, name))\n        complete.append(prefix)\n    return complete, incomplete\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--write_hashes", action="store_true")\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--splits_dir", required=True)\n    p.add_argument("--out", default="split_hashes.json")\n    p.add_argument("--input_root", default="/kaggle/input")\n    p.add_argument("--out_dir", default="runs")\n    p.add_argument("--folds", default="0,1,2,3,4")\n    p.add_argument("--allow_hash_mismatch", action="store_true")\n    a = p.parse_args()\n    local = split_hashes(a.index_csv, a.splits_dir)\n    if a.write_hashes:\n        json.dump(local, open(a.out, "w"), indent=1)\n        print("wrote", a.out, local)\n        return\n    seen, problems = check_hashes(a.input_root, local)\n    print(f"split_hashes.json files found: {seen}")\n    if problems and not a.allow_hash_mismatch:\n        raise SystemExit("SPLIT MISMATCH -- do not merge:\\n  " + "\\n  ".join(problems))\n    complete, incomplete = collect(a.input_root, a.out_dir, tuple(int(x) for x in a.folds.split(",")))\n    print("complete runs :", complete)\n    print("incomplete    :", incomplete, "(skipped)")\n    if not complete:\n        raise SystemExit("no complete run found under " + a.input_root)\n    open(os.path.join(a.out_dir, "RUNS.txt"), "w").write(",".join(complete))\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/merge_runs.py")

wrote src/merge_runs.py


## Environment

In [13]:
import sys, os
sys.path.insert(0, "/kaggle/working")
os.environ["UNIFESP_DATA_ROOT"] = IMAGES_DATASET
os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/working"
os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"
os.environ["UNIFESP_CACHE_DIR"] = "/kaggle/working/image_cache"
!pip install -q timm albumentations iterative-stratification pydicom
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(), "| torch", torch.__version__)

CUDA: True | GPUs: 2 | torch 2.11.0+cu130


## Index, patient-level splits, held-out set, image cache

In [14]:
!cd /kaggle/working && python -m src.build_index
!cd /kaggle/working && python -m src.make_splits
!cd /kaggle/working && python -m src.make_holdout_cv_splits --holdout_fold 4
!cd /kaggle/working && python -m src.merge_runs --write_hashes --index_csv index.csv --splits_dir splits --out split_hashes.json

Indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
train dcm files: 1738 | test dcm files: 743
Saved index to /kaggle/working/index.csv | rows: 2481

Unique patients -> train: 768, test: 330
Patient overlap between train/test (Kaggle's own split): 0
Train patients with >1 image: 483 / 768
Patients: 768 | images: 1738
Per-class positive patient count:
 label_0_Abdomen             55
label_1_Ankle               26
label_2_Cervical_Spine      46
label_3_Chest              457
label_4_Clavicles            5
label_5_Elbow               13
label_6_Feet                32
label_7_Finger              16
label_8_Forearm              9
label_9_Hand                42
label_10_Hip                18
label_11_Knee               44
label_12_Lower_Leg          13
label_13_Lumbar_Spine       47
label_14_Others             47
label_15_Pelvis             62
label_16_Shoulder           18
label_17_Sinus              15
label_18_Skull               5
la

In [15]:
!cd /kaggle/working && python -m src.cache_images

Indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
Found 2481 files. Caching to /kaggle/working/image_cache ...
  250/2481 (24.1s elapsed)
  500/2481 (46.7s elapsed)
  750/2481 (70.0s elapsed)
  1000/2481 (92.3s elapsed)
  1250/2481 (114.2s elapsed)
  1500/2481 (136.0s elapsed)
  1750/2481 (157.8s elapsed)
  2000/2481 (179.6s elapsed)
  2250/2481 (201.4s elapsed)
Done. Cached 2481 images in 221.6s -> /kaggle/working/image_cache


## Train every run on every CV fold (2 GPUs in parallel, resumable)

UNIFESP images are natively 256x256: nothing to gain by upsampling; Swin-T is fixed at 224.

In [16]:
import subprocess, itertools, threading, queue, os, time

EPOCHS, BATCH = 30, 32
RUNS = [  # (prefix, backbone, img_size, aug, lr)
    ('swin_tiny_s224', 'swin_tiny', 224, 'light', 0.0002),
    ('densenet121_s256', 'densenet121', 256, 'light', 0.0003),
    ('efficientnetv2_s_s256', 'efficientnetv2_s', 256, 'light', 0.0003),
    ('convnext_tiny_s256', 'convnext_tiny', 256, 'light', 0.0001),
    ('resnet50_s256_heavy', 'resnet50', 256, 'heavy', 0.0003),
    ('efficientnet_b3_s256_hv', 'efficientnet_b3', 256, 'heavy', 0.0003),
]
FOLDS = [0, 1, 2, 3, 4]
OUT = "/kaggle/working/runs"
CV = "/kaggle/working/splits/cv_folds.csv"
HOLD = "/kaggle/working/splits/holdout_uids.csv"

jobs = []
for (prefix, bb, size, aug, lr), f in itertools.product(RUNS, FOLDS):
    name = f"{prefix}_f{f}"
    if os.path.exists(f"{OUT}/{name}/preds.npz"):
        continue
    jobs.append((name, ["python", "-m", "src.train_v2", "--backbone", bb, "--img_size", str(size), "--aug", aug,
                        "--lr", str(lr), "--epochs", str(EPOCHS), "--batch_size", str(BATCH), "--fold", str(f),
                        "--folds_csv", CV, "--fold_col", "fold", "--holdout_csv", HOLD, "--run_name", name]))
print(len(jobs), "jobs to run")

q = queue.Queue()
for j in jobs: q.put(j)
failed = []

def worker(gpu):
    while True:
        try: name, cmd = q.get_nowait()
        except queue.Empty: return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        os.makedirs("/kaggle/working/logs", exist_ok=True)
        with open(f"/kaggle/working/logs/{name}.log", "w") as log:
            rc = subprocess.call(cmd, cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT)
        print(("[ok]   " if rc == 0 else "[FAIL] ") + name, time.strftime("%H:%M:%S"))
        if rc != 0: failed.append(name)

ths = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
[t.start() for t in ths]; [t.join() for t in ths]
print("failed:", failed)

30 jobs to run
[ok]   swin_tiny_s224_f0 20:11:20
[ok]   swin_tiny_s224_f1 20:11:20
[ok]   swin_tiny_s224_f2 20:16:20
[ok]   swin_tiny_s224_f3 20:16:26
[ok]   swin_tiny_s224_f4 20:21:22
[ok]   densenet121_s256_f0 20:22:12
[ok]   densenet121_s256_f1 20:27:14
[ok]   densenet121_s256_f2 20:27:56
[ok]   densenet121_s256_f3 20:33:07
[ok]   densenet121_s256_f4 20:33:50
[ok]   efficientnetv2_s_s256_f0 20:39:44
[ok]   efficientnetv2_s_s256_f1 20:40:25
[ok]   efficientnetv2_s_s256_f2 20:46:06
[ok]   efficientnetv2_s_s256_f3 20:46:55
[ok]   convnext_tiny_s256_f0 20:52:18
[ok]   efficientnetv2_s_s256_f4 20:52:42
[ok]   convnext_tiny_s256_f1 20:57:38
[ok]   convnext_tiny_s256_f2 20:57:58
[ok]   convnext_tiny_s256_f3 21:02:59
[ok]   convnext_tiny_s256_f4 21:03:18
[ok]   resnet50_s256_heavy_f0 21:07:53
[ok]   resnet50_s256_heavy_f1 21:08:11
[ok]   resnet50_s256_heavy_f2 21:12:45
[ok]   resnet50_s256_heavy_f3 21:13:03
[ok]   resnet50_s256_heavy_f4 21:17:26
[ok]   efficientnet_b3_s256_hv_f0 21:19:24
[o

## Ensemble + honest evaluation + Kaggle submission

Look at `holdout.ens_tuned_min_one` (the number to report), its CI, and `holdout.paired_ens_vs_single_sample_f1`.

In [17]:
import os
if "RUNS" in globals():     # single / part notebook: use only runs whose 5 folds all finished
    done = [r[0] for r in RUNS if all(os.path.exists(f"/kaggle/working/runs/{r[0]}_f{k}/preds.npz") for k in range(5))]
    print("complete runs :", done)
    print("incomplete (skipped; see /kaggle/working/logs/*.log):", [r[0] for r in RUNS if r[0] not in done])
    prefixes = ",".join(done)
else:                       # merge notebook: merge_runs already wrote the list of complete runs
    prefixes = open("/kaggle/working/runs/RUNS.txt").read()
assert prefixes, "no complete run -- check /kaggle/working/logs/*.log"
!cd /kaggle/working && python -m src.run_ensemble --out_dir runs --runs {prefixes} --folds 0,1,2,3,4 \
    --index_csv index.csv --cv_folds_csv splits/cv_folds.csv --holdout_csv splits/holdout_uids.csv \
    --sample_submission {IMAGES_DATASET}/sample_submission.csv --result_dir results
!cp /kaggle/working/results/submission.csv /kaggle/working/submission.csv

complete runs : ['swin_tiny_s224', 'densenet121_s256', 'efficientnetv2_s_s256', 'convnext_tiny_s256', 'resnet50_s256_heavy', 'efficientnet_b3_s256_hv']
incomplete (skipped; see /kaggle/working/logs/*.log): []
{
 "chosen_variant": {
  "swin_tiny_s224": "blend",
  "densenet121_s256": "blend",
  "efficientnetv2_s_s256": "blend_tta",
  "convnext_tiny_s256": "blend_tta",
  "resnet50_s256_heavy": "blend",
  "efficientnet_b3_s256_hv": "blend_tta"
 },
 "weights": {
  "swin_tiny_s224": 0.0,
  "densenet121_s256": 0.0,
  "efficientnetv2_s_s256": 0.0975609756097561,
  "convnext_tiny_s256": 0.2682926829268293,
  "resnet50_s256_heavy": 0.2926829268292683,
  "efficientnet_b3_s256_hv": 0.34146341463414637
 },
 "holdout": {
  "n_images": 323,
  "n_patients": 156,
  "ens_0.5": {
   "sample_mean_f1": 0.9164086687306502,
   "macro_f1": 0.8536512201107516,
   "micro_f1": 0.912981455064194,
   "subset_accuracy": 0.8885448916408669
  },
  "ens_0.5_min_one": {
   "sample_mean_f1": 0.9164086687306502,
   "macr

In [18]:
import json, pandas as pd
res = json.load(open("/kaggle/working/results/results.json"))
print(json.dumps({k: res[k] for k in ("weights", "chosen_variant")}, indent=1))
print(json.dumps(res["holdout"], indent=1)[:3500])
pd.read_csv("/kaggle/working/results/per_run.csv")

{
 "weights": {
  "swin_tiny_s224": 0.0,
  "densenet121_s256": 0.0,
  "efficientnetv2_s_s256": 0.0975609756097561,
  "convnext_tiny_s256": 0.2682926829268293,
  "resnet50_s256_heavy": 0.2926829268292683,
  "efficientnet_b3_s256_hv": 0.34146341463414637
 },
 "chosen_variant": {
  "swin_tiny_s224": "blend",
  "densenet121_s256": "blend",
  "efficientnetv2_s_s256": "blend_tta",
  "convnext_tiny_s256": "blend_tta",
  "resnet50_s256_heavy": "blend",
  "efficientnet_b3_s256_hv": "blend_tta"
 }
}
{
 "n_images": 323,
 "n_patients": 156,
 "ens_0.5": {
  "sample_mean_f1": 0.9164086687306502,
  "macro_f1": 0.8536512201107516,
  "micro_f1": 0.912981455064194,
  "subset_accuracy": 0.8885448916408669
 },
 "ens_0.5_min_one": {
  "sample_mean_f1": 0.9164086687306502,
  "macro_f1": 0.8536512201107516,
  "micro_f1": 0.912981455064194,
  "subset_accuracy": 0.8885448916408669
 },
 "ens_tuned_min_one": {
  "sample_mean_f1": 0.9188854489164086,
  "macro_f1": 0.8630176112953246,
  "micro_f1": 0.9147727272727

,run,chosen_variant,oof_sig,oof_soft,oof_blend,oof_sig_tta,oof_soft_tta,oof_blend_tta,holdout_sample_mean_f1_0.5_min_one
0,swin_tiny_s224,blend,0.912203,0.900589,0.914676,0.912815,0.900353,0.912674,0.913313
1,densenet121_s256,blend,0.909305,0.900824,0.917974,0.908316,0.901531,0.915618,0.922085
2,efficientnetv2_s_s256,blend_tta,0.913781,0.905771,0.916419,0.913192,0.904122,0.918634,0.917337
3,convnext_tiny_s256,blend_tta,0.919199,0.908834,0.922238,0.919859,0.908834,0.923416,0.912281
4,resnet50_s256_heavy,blend,0.875618,0.895289,0.906408,0.877789,0.897644,0.905324,0.910423
5,efficientnet_b3_s256_hv,blend_tta,0.910059,0.895524,0.911614,0.909144,0.893875,0.913263,0.917234


## Next

Submit `submission.csv` at <https://www.kaggle.com/competitions/unifesp-x-ray-body-part-classifier/submit> (Late Submission, ONE account). Leaderboard: #1 = 0.93378, #2 = 0.92787; the private score uses ~60 % of the test set with the original (noisy) labels, so ~0.003 differences are noise.